# Ablation Experiment - SIAB Dataset

Our goal is to determine the stability/consistency of multiverse decision importances when different decisions are fixed to a particular decision.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import sys
from pathlib import Path
# metrics.py lives at ../../../src/metrics/metrics.py relative to this notebook
sys.path.insert(0, str((Path("../../../src/metrics")).resolve()))
from metrics import (
    organize_multiverse_results,
    within_bucket_variation,
    total_variance_decomposition,
)

# get all csv files in the data directory for run of interest
data_dir = Path("../output/runs/1/data")
csv_files = list(data_dir.glob("*.csv"))
print(f"Found {len(csv_files)} CSV files.")

# read all files with mixed types
arrays_lst = []
arrays = []
for i, f in enumerate(csv_files):
    arr = np.genfromtxt(f, delimiter=',', dtype=None, encoding='utf-8')
    arrays.append(arr)
    if (i + 1) % 1000 == 0:
        arrays_lst.append(np.concatenate(arrays))
        arrays = []
        print(f"Loaded {i+1}/{len(csv_files)} files")
arrays_lst.append(np.concatenate(arrays))
# stack all arrays vertically
data = np.concatenate(arrays_lst)
# convert data to pandas df
df = pd.DataFrame(data)
# remove all rows that are exact duplicates, since they are removing the rows of column names
df = df.drop_duplicates()
# make first row the header
df.columns = df.iloc[0]
df = df[1:].reset_index(drop=True)

In [ ]:
# look at data
print(df.shape)
df.head()

In [ ]:
# set up inputs for `organize_multiverse_results`
dimension_order = ["spell_gap_threshold",
 "new_job_notif_threshold",
 "new_job_no_notif_threshold",
 "missing_strategy",
 "split_seed",
 "model"]
# prepend mv_dim_ to all dimension names
dimension_order = ["mv_dim_" + dim for dim in dimension_order]
performance_metrics = ["acc", "f1", "roc_auc"]
# prepend test_ to all performance metric names
performance_metrics = ["test_" + metric for metric in performance_metrics]

# make all cols that start with test_ numeric
for col in df.columns:
    if col.startswith("test_"):
        df[col] = pd.to_numeric(df[col])

In [ ]:
# call helper function to make a structured df
df = organize_multiverse_results(df, dimension_order, performance_metrics)

In [ ]:
# ensure types are correct
print(df.dtypes)
df.head()

In [ ]:
# conduct analyses with full multiverse
full_wuv = within_bucket_variation(df, "test_acc")
full_buv = total_variance_decomposition(df, "test_acc")
full_buv = {key: full_buv[key]['expected_var_within_groups'] for key in full_buv.keys()}

# compute regression coefficients and r^2
dimension_cols = [col for col in df.columns if col[1] == "dimension"]
X = df[dimension_cols].copy()
# remove the multiindex and just keep the dimension name as the column name
X = X.droplevel(level=1, axis=1)
X = pd.get_dummies(X, columns=X.columns)
y = df["test_acc"].astype(float)
reg = LinearRegression()
reg.fit(X, y)
full_coefs = {X.columns[i]: reg.coef_[0, i] for i in range(len(X.columns))}
full_r2 = r2_score(y, reg.predict(X))

In [ ]:
# set "default" value for each dimension
default_values = {
    "mv_dim_spell_gap_threshold": "42",
    "mv_dim_new_job_notif_threshold": "92",
    "mv_dim_new_job_no_notif_threshold": "366",
    "mv_dim_missing_strategy": "all_impute_median",
    "mv_dim_split_seed": "0",
    "mv_dim_model": "random_forest",
}

In [ ]:
buv_ablation = {}
wuv_ablation = {}
fanova_ablation = {}
regression_ablation = {}
for dim, default_val in default_values.items():
    
    # get df with only the default value for this dimension
    temp_df = df[df[(dim, "dimension")] == default_val]
    
    # compute WUV
    wuv_ablation[dim] = within_bucket_variation(temp_df, "test_acc")
    
    # compute BUV
    buv = total_variance_decomposition(temp_df, "test_acc")
    buv = {key: buv[key]['expected_var_within_groups'] for key in buv.keys()}
    buv_ablation[dim] = buv
    
    # compute regression coefficients and r^2
    dimension_cols = [col for col in temp_df.columns if col[1] == "dimension"]
    X = temp_df[dimension_cols].copy()
    # remove the multiindex and just keep the dimension name as the column name
    X = X.droplevel(level=1, axis=1)
    X = pd.get_dummies(X, columns=X.columns)
    print(f"Shape of temp_df: {temp_df.shape}, Shape of X: {X.shape}")
    y = temp_df["test_acc"].astype(float)
    reg = LinearRegression()
    reg.fit(X, y)
    coefs = {X.columns[i]: reg.coef_[0, i] for i in range(len(X.columns))}
    r2 = r2_score(y, reg.predict(X))
    regression_ablation[dim] = {'coefficients': coefs, 'r2': r2}

In [ ]:
print(wuv_ablation)
print('---------------')
print(buv_ablation)
print('---------------')
print(regression_ablation)

In [ ]:
# Short labels for display, preserving dimension_order ordering
DIM_LABELS = [d.replace("mv_dim_", "") for d in dimension_order]

def normalize(d):
    total = sum(d.values()) or 1
    return {k: v / total for k, v in d.items()}

def pct_change(ablated_val, full_val):
    return (ablated_val - full_val) / abs(full_val) * 100 if full_val != 0 else np.nan

def aggregate_reg_importance(coef_dict, dims, agg="sum"):
    """Sum or mean of |coef| for all one-hot dummies belonging to each dimension."""
    result = {}
    for dim in dims:
        vals = [abs(v) for k, v in coef_dict.items() if k.startswith(f"mv_dim_{dim}_")]
        result[dim] = sum(vals) if agg == "sum" else (sum(vals) / len(vals) if vals else 0.0)
    return result

In [ ]:
# Normalize full WUV so columns are on the same scale
full_wuv_norm = normalize({d.replace("mv_dim_", ""): v for d, v in full_wuv.items()})
full_buv_named = {d.replace("mv_dim_", ""): v for d, v in full_buv.items()}
full_reg_sum = aggregate_reg_importance(full_coefs, DIM_LABELS, agg="sum")
full_reg_mean = aggregate_reg_importance(full_coefs, DIM_LABELS, agg="mean")

wuv_pct, buv_pct, reg_sum_pct, reg_mean_pct = {}, {}, {}, {}

for dim_full, dim_label in zip(default_values.keys(), DIM_LABELS):
    col = f"fix:\n{dim_label}"

    ab_wuv_norm = normalize({d.replace("mv_dim_", ""): v for d, v in wuv_ablation[dim_full].items()})
    wuv_pct[col] = {d: pct_change(ab_wuv_norm[d], full_wuv_norm[d]) for d in DIM_LABELS}

    ab_buv = {d.replace("mv_dim_", ""): v for d, v in buv_ablation[dim_full].items()}
    buv_pct[col] = {d: pct_change(ab_buv[d], full_buv_named[d]) for d in DIM_LABELS}

    ab_reg_sum = aggregate_reg_importance(regression_ablation[dim_full]["coefficients"], DIM_LABELS, agg="sum")
    reg_sum_pct[col] = {d: pct_change(ab_reg_sum[d], full_reg_sum[d]) for d in DIM_LABELS}

    ab_reg_mean = aggregate_reg_importance(regression_ablation[dim_full]["coefficients"], DIM_LABELS, agg="mean")
    reg_mean_pct[col] = {d: pct_change(ab_reg_mean[d], full_reg_mean[d]) for d in DIM_LABELS}

wuv_pct_df = pd.DataFrame(wuv_pct, index=DIM_LABELS)
buv_pct_df = pd.DataFrame(buv_pct, index=DIM_LABELS)
reg_sum_pct_df = pd.DataFrame(reg_sum_pct, index=DIM_LABELS)
reg_mean_pct_df = pd.DataFrame(reg_mean_pct, index=DIM_LABELS)

In [ ]:
def plot_pct_heatmap(df, title, fmt="+.1f"):
    fig, ax = plt.subplots(figsize=(10, 4))
    vals = df.values.astype(float)
    masked = np.ma.masked_invalid(vals)
    cmap_obj = plt.get_cmap("RdBu_r").copy()
    cmap_obj.set_bad(color="#cccccc")
    absmax = np.nanmax(np.abs(vals))
    im = ax.imshow(masked, aspect="auto", cmap=cmap_obj, vmin=-absmax, vmax=absmax)
    plt.colorbar(im, ax=ax, fraction=0.02, pad=0.02, label="% change from full")
    ax.set_xticks(range(len(df.columns)))
    ax.set_xticklabels(df.columns, fontsize=8)
    ax.set_yticks(range(len(df.index)))
    ax.set_yticklabels(df.index, fontsize=9)
    ax.set_title(title, fontsize=11, pad=10)
    for r in range(vals.shape[0]):
        for c in range(vals.shape[1]):
            v = vals[r, c]
            text = f"{v:{fmt}}%" if not np.isnan(v) else "N/A"
            ax.text(c, r, text, ha="center", va="center", fontsize=7, color="black")
    fig.tight_layout()
    plt.show()

plot_pct_heatmap(wuv_pct_df, "WUV importance % change (normalized) when each dimension is fixed")
plot_pct_heatmap(buv_pct_df, "BUV importance % change when each dimension is fixed")
plot_pct_heatmap(reg_sum_pct_df, "Regression sum(|coef|) importance % change when each dimension is fixed")
plot_pct_heatmap(reg_mean_pct_df, "Regression mean(|coef|) importance % change when each dimension is fixed")

In [ ]:
# Build raw importance DataFrames (full + each ablated setting)
wuv_raw = {"full": full_wuv_norm}
buv_raw = {"full": full_buv_named}
reg_sum_raw = {"full": full_reg_sum}
reg_mean_raw = {"full": full_reg_mean}

for dim_full, dim_label in zip(default_values.keys(), DIM_LABELS):
    col = f"fix:\n{dim_label}"
    wuv_raw[col] = normalize({d.replace("mv_dim_", ""): v for d, v in wuv_ablation[dim_full].items()})
    buv_raw[col] = {d.replace("mv_dim_", ""): v for d, v in buv_ablation[dim_full].items()}
    reg_sum_raw[col] = aggregate_reg_importance(regression_ablation[dim_full]["coefficients"], DIM_LABELS, agg="sum")
    reg_mean_raw[col] = aggregate_reg_importance(regression_ablation[dim_full]["coefficients"], DIM_LABELS, agg="mean")

wuv_raw_df = pd.DataFrame(wuv_raw, index=DIM_LABELS)
buv_raw_df = pd.DataFrame(buv_raw, index=DIM_LABELS)
reg_sum_raw_df = pd.DataFrame(reg_sum_raw, index=DIM_LABELS)
reg_mean_raw_df = pd.DataFrame(reg_mean_raw, index=DIM_LABELS)

# Rank within each column (rank 1 = highest importance)
wuv_rank_df = wuv_raw_df.rank(ascending=False, method="min").astype(int)
buv_rank_df = buv_raw_df.rank(ascending=False, method="min").astype(int)
reg_sum_rank_df = reg_sum_raw_df.rank(ascending=False, method="min").astype(int)
reg_mean_rank_df = reg_mean_raw_df.rank(ascending=False, method="min").astype(int)

In [ ]:
def plot_rank_heatmap(df, title):
    fig, ax = plt.subplots(figsize=(11, 4))
    vals = df.values.astype(float)
    cmap_obj = plt.get_cmap("YlOrRd_r").copy()
    im = ax.imshow(vals, aspect="auto", cmap=cmap_obj, vmin=1, vmax=len(df.index))
    plt.colorbar(im, ax=ax, fraction=0.02, pad=0.02, label="rank (1 = most important)")
    ax.set_xticks(range(len(df.columns)))
    ax.set_xticklabels(df.columns, fontsize=8)
    ax.set_yticks(range(len(df.index)))
    ax.set_yticklabels(df.index, fontsize=9)
    ax.set_title(title, fontsize=11, pad=10)
    for r in range(vals.shape[0]):
        for c in range(vals.shape[1]):
            ax.text(c, r, str(int(vals[r, c])), ha="center", va="center", fontsize=9, color="black")
    fig.tight_layout()
    plt.show()

plot_rank_heatmap(wuv_rank_df, "WUV importance ranking per setting (1 = most important)")
plot_rank_heatmap(buv_rank_df, "BUV importance ranking per setting (1 = most important)")
plot_rank_heatmap(reg_sum_rank_df, "Regression sum(|coef|) importance ranking per setting (1 = most important)")
plot_rank_heatmap(reg_mean_rank_df, "Regression mean(|coef|) importance ranking per setting (1 = most important)")